# Step 3 — Single Agent

🇬🇧 **English** (this notebook)

Move from a hand-written prompt to a CrewAI `Agent`. This notebook is **standalone** — the agent is defined right here in code, no separate YAML config or project files to edit. The `role`, `goal`, and `backstory` are still just a system prompt under the hood — CrewAI assembles it for you. What the framework adds is the loop: the agent reasons in steps before producing output, can call tools (steps 5a/5b), and retries on failure.

## Background

The core loop that makes an agent an agent — alternating between reasoning about what to do next and taking an action (calling a tool, reading a result, updating a plan) — was introduced in:

> Yao, S., Zhao, J., Yu, D., Du, N., Shafran, I., Narasimhan, K., & Cao, Y. (2022). *ReAct: Synergizing Reasoning and Acting in Language Models*. ICLR 2023. [arXiv:2210.03629](https://arxiv.org/abs/2210.03629)

ReAct (Reason + Act) is the pattern CrewAI agents follow: the model thinks ("I need to find X"), acts (calls a tool), observes the result, thinks again, and repeats until it can produce a final answer. This is what separates an agent from a single prompt call — the loop.

The broader observation that LLM-based agents benefit from an explicit module structure — profile (who the agent is), memory, planning, action — was systematized in:

> Wang, L., Ma, C., Feng, X., Zhang, Z., Yang, H., Zhang, J., Chen, Z., Tang, J., Chen, X., Lin, Y., Zhao, W. X., Wei, Z., & Wen, J. (2023). *A Survey on Large Language Model based Autonomous Agents*. [arXiv:2308.11432](https://arxiv.org/abs/2308.11432)

![Unified framework for LLM-based autonomous agents: Profile, Memory, Planning, Action modules](../assets/agentsurvey-wang2023-fig2.png)
*Figure 2 from Wang et al. (2023). Reproduced for educational use in this course.*

In CrewAI terms: `role`/`goal`/`backstory` = **Profile**; `tools` + the ReAct loop inside `kickoff()` = **Action**.

## The exercise

`Agent.kickoff(...)` runs a single agent standalone — no `Crew`, no `Task`, no YAML files needed. Just define the agent's identity and call it with a question.

The worked example below runs a single **Researcher** agent — the same role this repo's full demo crew uses (see `src/research_crew/config/agents.yaml`) — investigating the EU AI Act. Steps 4–5c reuse this exact case, each adding one capability, so you can compare what each addition actually changes.

In [1]:
import os

from dotenv import load_dotenv
from crewai import Agent

load_dotenv()

# ── Agent identity — the same "researcher" template as
# src/research_crew/config/agents.yaml, with {topic} filled in via an
# f-string instead of CrewAI's own YAML interpolation ──────────────────────
topic = "EU AI Act"

role      = f"{topic} Senior Data Researcher"
goal      = f"Uncover cutting-edge developments in {topic}"
backstory = (
    f"You're a seasoned researcher with a knack for uncovering the latest "
    f"developments in {topic}. Known for your ability to find the most relevant "
    f"information and present it in a clear and concise manner."
)

agent = Agent(
    role=role,
    goal=goal,
    backstory=backstory,
    verbose=True,
)

# ── The question — same topic as previous steps ──────────────────────────────
question = (
    f"Explain {topic}'s risk-based categories (unacceptable, high-risk, "
    "limited, minimal) and what obligations apply to providers of high-risk AI systems."
)

# Jupyter runs its own event loop, and kickoff() detects that automatically and
# returns a coroutine instead of running synchronously - awaiting it is required here
# (this quirk only shows up in a notebook; a plain .py script would not need it).
result = await agent.kickoff(question)
print(result.raw)

╭───────────────────────────────────────────── 🤖 LiteAgent Started ──────────────────────────────────────────────╮
│                                                                                                                 │
│  LiteAgent Started                                                                                              │
│  Role: EU AI Act Senior Data Researcher                                                                         │
│  Status: In Progress                                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── 🤖 LiteAgent Started ──────────────────────────────────────────────╮
│                                                                                                                 │
│  LiteAgent Session Started                                                                                      │
│  Name:                                                                                                          │
│  EU AI Act Senior Data Researcher                                                                               │
│  id:                                                                                                            │
│  91963bde-f775-463c-8a23-ffec60aafbc4                                                                           │
│  role:                                                                                                          │
│  EU AI Act Senior Data Researcher                                                                               │
│  goal:                                                                                                          │
│  Uncover cutting-edge developments in EU AI Act                                                                 │
│  backstory:                                                                                                     │
│  You're a seasoned researcher with a knack for uncovering the latest developments in EU AI Act. Known for your  │
│  ability to find the most relevant information and present it in a clear and concise manner.                    │
│  tools:                                                                                                         │
│  []                                                                                                             │
│  verbose:                                                                                                       │
│  True                                                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: EU AI Act Senior Data Researcher                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  As a senior researcher tracking the regulatory landscape of the EU AI Act (Regulation (EU) 2024/1689), I have  │
│  synthesized the core architecture of the Act’s risk-based approach.                                            │
│                                                                                                                 │
│  The EU AI Act classifies AI systems based on the level of risk they pose to the health, safety, and            │
│  fundamental rights of citizens. The regulation operates on a "pyramid" structure, where requirements become    │
│  progressively more stringent as the risk profile increases.                                                    │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### The Four Risk Categories                                                                                   │
│                                                                                                                 │
│  1.  **Unacceptable Risk (Prohibited):**                                                                        │
│      These systems are deemed a clear threat to fundamental rights and are **banned**. This includes cognitive  │
│  behavioral manipulation, social scoring by governments, real-time remote biometric identification in public    │
│  spaces (with narrow exceptions for law enforcement), and AI used to exploit the vulnerabilities of specific    │
│  groups (children, people with disabilities).                                                                   │
│                                                                                                                 │
│  2.  **High-Risk (Strictly Regulated):**                                                                        │
│      These systems are permitted but must comply with a rigorous set of legal requirements before hitting the   │
│  market. This category includes AI used in critical infrastructure, education, employment, essential            │
│  private/public services (e.g., credit scoring, healthcare), and law enforcement.                               │
│                                                                                                                 │
│  3.  **Limited Risk (Transparency):**                                                                           │
│      These systems are subject to specific **transparency obligations**. Users must be informed that they are   │
│  interacting with an AI system (e.g., chatbots, emotion recognition systems, or AI-generated deepfakes). This   │
│  ensures that individuals can make informed decisions about their engagement with the technology.               │
│                                                                                                                 │
│  4.  **Minimal/No Risk:**                                                                                       │
│      The vast majority of AI systems in the EU (such as spam filters, AI-enabled video games, or inventory      │
│  management) fall into this category. They are essentia

As a senior researcher tracking the regulatory landscape of the EU AI Act (Regulation (EU) 2024/1689), I have synthesized the core architecture of the Act’s risk-based approach. 

The EU AI Act classifies AI systems based on the level of risk they pose to the health, safety, and fundamental rights of citizens. The regulation operates on a "pyramid" structure, where requirements become progressively more stringent as the risk profile increases.

---

### The Four Risk Categories

1.  **Unacceptable Risk (Prohibited):**
    These systems are deemed a clear threat to fundamental rights and are **banned**. This includes cognitive behavioral manipulation, social scoring by governments, real-time remote biometric identification in public spaces (with narrow exceptions for law enforcement), and AI used to exploit the vulnerabilities of specific groups (children, people with disabilities).

2.  **High-Risk (Strictly Regulated):**
    These systems are permitted but must comply with a rigorous 

╭──────────────────────────────────────────── ✅ LiteAgent Completed ─────────────────────────────────────────────╮
│                                                                                                                 │
│  LiteAgent Completed                                                                                            │
│  Role: EU AI Act Senior Data Researcher                                                                         │
│  id: 91963bde-f775-463c-8a23-ffec60aafbc4                                                                       │
│  role: EU AI Act Senior Data Researcher                                                                         │
│  goal: Uncover cutting-edge developments in EU AI Act                                                           │
│  backstory: You're a seasoned researcher with a knack for uncovering the latest developments in EU AI Act.      │
│  Known for your ability to find the most relevant information and present it in a clear and concise manner.     │
│  tools: []                                                                                                      │
│  verbose: True                                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Your task

1. Run the cell. Read the verbose log above the final answer — this is the first time you can see the agent's internal reasoning, not just the final answer. Does the agent break the task into sub-steps?

2. Compare this Researcher's answer to step 2's plain-prompt answer on a similar question. What does having an explicit `role`/`goal`/`backstory` add, if anything, over a system message you wrote by hand — given that CrewAI compiles them into almost the same thing under the hood (see the Background note above)?

3. Now swap in your own team's topic: replace `role`, `goal`, `backstory`, and `question` with an agent suited to your topic. Keep that identity the same across steps 4–5c so the comparisons stay meaningful.

4. Fill in the **Step 3** section of `EVALUATION.md`.

## Stretch goal

Look at the verbose log's "Final Answer" alongside the agent's intermediate reasoning. Find one place where the reasoning and the conclusion seem inconsistent. What does this tell you about trusting chain-of-thought?

---

**→ Interim submission is due after this step.** See [Assignment Overview](../../team_assignment/en/assignment-overview.md) for exactly what's expected.